# 🎬 Movie Recommendation System using SVD

This notebook builds a movie recommender system using collaborative filtering (SVD) on the MovieLens dataset.

## Objectives
- Learn user preferences from ratings
- Predict ratings for unseen movies
- Recommend top movies


In [19]:
!pip install "numpy<2"
!pip install scikit-surprise

In [20]:
import pandas as pd
from surprise import Dataset, SVD
from surprise.model_selection import train_test_split


## Load MovieLens Dataset
We use the built-in MovieLens 100k dataset.

## Model Evaluation vs Recommendation

This notebook follows a two-stage approach:

1. **Evaluation Stage**
   - The data is split into training and test sets
   - The model is trained on training data and evaluated on unseen test data

2. **Recommendation Stage**
   - After evaluation, the model is retrained on the full dataset
   - This allows the system to generate better recommendations using all available data

This mirrors real-world recommender systems where models are validated first, then retrained for deployment.

In [21]:
data= Dataset.load_builtin("ml-100k")
trainset,testset=train_test_split(data,test_size=0.2)


## Load Movie Metadata
We map movie IDs to movie names.

In [22]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k/u.item

--2026-05-09 02:06:24--  https://files.grouplens.org/datasets/movielens/ml-100k/u.item
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 236344 (231K)
Saving to: ‘u.item.1’

u.item.1            100%[===================>] 230.80K  --.-KB/s    in 0.09s   

2026-05-09 02:06:24 (2.59 MB/s) - ‘u.item.1’ saved [236344/236344]



In [23]:

movies_df = pd.read_csv(
    "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1]
)
movies_df.columns = ["movieId", "title"]

movie_dict = dict(zip(movies_df.movieId, movies_df.title))


## Train SVD Model
We use matrix factorization to learn user and movie embeddings.

In [24]:
model=SVD()
model.fit(trainset)


## Model Evaluation (RMSE & MAE)

We evaluate how accurately the model predicts ratings using:
- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)

In [25]:

from surprise import accuracy

# predict on test set
predictions = model.test(testset)

# evaluate
print("Evaluation Results:")
accuracy.rmse(predictions)
accuracy.mae(predictions)

Evaluation Results:
RMSE: 0.9382
MAE:  0.7377


0.737686039008378


### Interpretation

- RMSE measures how far predicted ratings are from actual ratings
- MAE measures average absolute error

Lower values indicate better prediction accuracy.
``


## Recommendation Quality (Precision@K)

Precision@K measures how many of the top-K recommended items are actually relevan

In [26]:
from collections import defaultdict

def precision_at_k(predictions, k=5, threshold=4):
    user_est_true = defaultdict(list)

    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        top_k = user_ratings[:k]

        relevant = sum((true_r >= threshold) for (_, true_r) in top_k)

        precisions[uid] = relevant / k

    return sum(precisions.values()) / len(precisions)

In [27]:

precision = precision_at_k(predictions, k=5)

print(f"Precision@5: {precision:.4f}")

Precision@5: 0.6945


### Evaluation Summary

- RMSE / MAE → measure rating prediction accuracy
- Precision@K → measures recommendation quality

In real-world systems, ranking-based metrics like Precision@K are more important.

## Final Model for Recommendations

After evaluating the model, we retrain it on the full dataset.

This ensures:
- all user interactions are used
- better recommendation quality

In [28]:
# build full dataset for recommendations
full_trainset = data.build_full_trainset()

# train final model
final_model = SVD()
final_model.fit(full_trainset)

# select a valid user
sample_uid = next(iter(full_trainset.all_users()))
user_id = full_trainset.to_raw_uid(sample_uid)

print("Generating recommendations for user:", user_id)

Generating recommendations for user: 196



## Inference Time - Generate Recommendations
We recommend movies the user hasn’t already rated.

In [29]:
# find seen movies
seen_movies = set()

for (uid, iid, _) in full_trainset.all_ratings():
    if uid == full_trainset.to_inner_uid(user_id):
        seen_movies.add(int(full_trainset.to_raw_iid(iid)))

# get all movies
movie_ids = set(movies_df.movieId)

# get unseen movies
unseen_movies = movie_ids - seen_movies

# generate predictions
predictions = []

for m in unseen_movies:
    pred = final_model.predict(user_id, str(m))
    predictions.append((m, pred.est))

# sort by predicted rating
predictions.sort(key=lambda x: x[1], reverse=True)

## Top Recommendations



In [30]:

print("Top 5 recommendations:\n")

for i, (movie_id, score) in enumerate(predictions[:5], 1):
    movie_name = movie_dict.get(movie_id, "Unknown Movie")
    print(f"{i}. {movie_name} — Predicted Rating: {score:.2f}")


Top 5 recommendations:

1. Close Shave, A (1995) — Predicted Rating: 4.54
2. North by Northwest (1959) — Predicted Rating: 4.53
3. Wings of Desire (1987) — Predicted Rating: 4.50
4. Raiders of the Lost Ark (1981) — Predicted Rating: 4.50
5. Wallace & Gromit: The Best of Aardman Animation (1996) — Predicted Rating: 4.47


## Inspect Learned Embeddings

In [31]:
print("User embedding (first 5 values):")
print(model.pu[0][:5])

print("\nMovie embedding (first 5 values):")
print(model.qi[0][:5])

User embedding (first 5 values):
[ 0.26338644 -0.13999262 -0.0645019  -0.01940796 -0.0112626 ]

Movie embedding (first 5 values):
[ 0.14160243 -0.14192236 -0.42889698  0.07995598 -0.10998939]


## Conclusion

- Built a recommender system using SVD
- Learned latent user and movie features
- Generated personalized recommendations
